# LC 56 — Merge Intervals
**Difficulty:** Medium | **Category:** Greedy | **Pattern:** Sort + Sweep Merge

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Sort by start time. Then walk forward
with one "current" interval. If the next interval overlaps (its start
is ≤ current end), extend the current end. If not, seal the current
interval and start a new one. Sorting is the only hard step.
</div>

## Official Problem Statement

Given an array of `intervals` where `intervals[i] = [start_i, end_i]`,
merge all overlapping intervals, and return an array of the
non-overlapping intervals that cover all the intervals in the input.

**Constraints:**
- `1 <= intervals.length <= 10^4`
- `intervals[i].length == 2`
- `0 <= start_i <= end_i <= 10^4`

## What This Is Actually Asking

You have a list of time ranges (or number ranges). Some of them
overlap or touch. You want to combine all overlapping ones into
single continuous ranges.

The output should have no two intervals that share any point.
The order of the input does not matter — you sort first.
Two intervals [1,3] and [3,5] are considered overlapping because
they share the endpoint 3.

## Walk Through an Example by Hand

`intervals = [[1,3],[2,6],[8,10],[15,18]]`

Step 1 — Sort by start (already sorted here):
```
[[1,3], [2,6], [8,10], [15,18]]
```

Step 2 — Initialize cur = [1, 3]

```
Next=[2,6]:  2 <= 3 → overlap! cur=[1, max(3,6)]=[1,6]
Next=[8,10]: 8 > 6  → no overlap. Append [1,6]. cur=[8,10]
Next=[15,18]:15 > 10 → no overlap. Append [8,10]. cur=[15,18]
End of list: Append [15,18]
```

Result: `[[1,6],[8,10],[15,18]]`

## The Picture

```
Input intervals on a timeline (0–18):

[1,3]    ├──┤
[2,6]      ├────┤
[8,10]           ├──┤
[15,18]                  ├───┤
         0  2  4  6  8  10  12  14  16  18

After sorting by start:
[1,3]  and [2,6] overlap → merge to [1,6]
[1,6]    ├──────┤
[8,10]           ├──┤         no overlap → keep separate
[15,18]                  ├───┤ no overlap → keep separate

Merged result:
[1,6]    ├──────┤
[8,10]           ├──┤
[15,18]                  ├───┤

Key: overlap condition is  next_start <= cur_end
     merge by taking       cur_end = max(cur_end, next_end)
```

## When To Use This Pattern

- When you see **"merge overlapping intervals"**, think
  *sort by start, then sweep with a current interval*.
- When you see **"consolidate time windows"**, think
  *sort + greedy merge*.
- When next interval's start is **≤ current end**, think
  *extend current end, do not close yet*.
- When next interval's start is **> current end**, think
  *seal current, begin tracking new interval*.
- When asked to **count gaps or free slots**, think
  *merge first, then examine spaces between merged intervals*.

## The Approach

Sort all intervals by their start value. Initialize a `cur` variable
to the first interval. Walk through the remaining intervals one by
one. If the next interval's start is at or before `cur`'s end, they
overlap — update `cur`'s end to the maximum of the two ends. If there
is no overlap, append `cur` to the result and set `cur` to the next
interval. After the loop, append the final `cur`. Return the result.

In [ ]:
from typing import List  # type hints for function signatures

In [ ]:
def test_harness(func):
    """Run all test cases against func and report results."""
    def norm(intervals):
        """Sort outer list and each inner pair for comparison."""
        return sorted([sorted(iv) for iv in intervals])

    tests = [
        # (input_intervals, expected)
        (
            [[1,3],[2,6],[8,10],[15,18]],
            [[1,6],[8,10],[15,18]]
        ),
        (
            [[1,4],[4,5]],
            [[1,5]]           # touching endpoints merge
        ),
        (
            [[1,4]],
            [[1,4]]           # single interval
        ),
        (
            [[1,4],[2,3]],
            [[1,4]]           # one fully inside another
        ),
        (
            [[1,2],[3,4],[5,6]],
            [[1,2],[3,4],[5,6]]  # no overlaps
        ),
        (
            [[2,6],[1,3],[8,10],[15,18]],
            [[1,6],[8,10],[15,18]]  # unsorted input
        ),
        (
            [[1,10],[2,3],[4,5],[6,7]],
            [[1,10]]          # all absorbed by first
        ),
    ]
    passed = 0
    for intervals, expected in tests:
        result = func([iv[:] for iv in intervals])  # pass copy
        if norm(result) == norm(expected):
            passed += 1
        else:
            print(f"FAILED | input={intervals}")
            print(f"        expected={expected}, got={result}")
    total = len(tests)
    print(f"\n{passed}/{total} tests passed.")

In [ ]:
def merge(intervals: List[List[int]]) -> List[List[int]]:
    """
    Sort by start, then sweep and merge overlapping intervals.

    Two intervals overlap when next_start <= cur_end.
    Merge by extending cur_end = max(cur_end, next_end).

    Args:
        intervals: List of [start, end] pairs (any order).
    Returns:
        List of merged non-overlapping [start, end] pairs.

    Time:  O(n log n)  — dominated by sort
    Space: O(n)        — output list
    """
    pass


# --- Debug prints (remove before final submission) ---
iv1 = [[1,3],[2,6],[8,10],[15,18]]
print(merge(iv1))   # expected: [[1,6],[8,10],[15,18]]

iv2 = [[1,4],[4,5]]
print(merge(iv2))   # expected: [[1,5]]

iv3 = [[1,4],[2,3]]
print(merge(iv3))   # expected: [[1,4]]

iv4 = [[2,6],[1,3],[8,10]]
print(merge(iv4))   # expected: [[1,6],[8,10]]

iv5 = [[1,10],[2,3],[4,5]]
print(merge(iv5))   # expected: [[1,10]]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(merge)

## Complexity

| Approach              | Time       | Space |
|-----------------------|------------|-------|
| Brute Force (compare all pairs) | O(n²) | O(n)  |
| Sort + Sweep Merge    | O(n log n) | O(n)  |

## Real World Connection

At Citi, maintenance windows for trading systems must be consolidated
before scheduling. Overlapping downtime windows from different teams
are merged so the ops team sees a single clean list of outage blocks.

In AWS Glue and EMR, ETL job partitions often produce overlapping
date ranges due to reprocessing. Before writing to S3 or Redshift,
the pipeline merges these ranges to avoid duplicate data.

In data engineering more broadly, SLA windows from upstream sources
overlap when systems retry. Merging intervals lets schedulers
calculate the true blocked time and identify free slots for new jobs.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra